# Silver weekly rosters

## Pipeline overview

**Purpose:** Preserve nflverse weekly roster snapshots as clean, source-shaped Silver records. A roster row describes a player on a team's roster in a scheduled week; even `ACT` status does not prove game participation. This notebook does not change existing consumers or build fantasy features.

| Role | Files | Columns | Grain |
| --- | --- | ---: | --- |
| Data input | `data/bronze/rosters_weekly/rosters_weekly_{season}.parquet`, 2023–2026 | 36 per season | Source roster player-team-week |
| Validation only | `data/silver/schedules/silver_schedules_{season}.parquet` | 46 per season | `game_id` |
| Output | `data/silver/rosters_weekly/silver_rosters_weekly_{season}.parquet` | 36 per season | `gsis_it_id + season + week + team` |

**Cleaning:** Remove exact full-row duplicates; fail on conflicting roster keys; normalize text and five whole-valued fields; preserve all source names, values, nulls, positions, and statuses. `gsis_it_id` is a source text identifier and the complete roster key. Nullable `gsis_id` remains the exact cross-source player ID; it is never filled by name or alternate ID.

**Blocking checks:** Schema and season; grain uniqueness; safe type conversion; stable populated `gsis_it_id` → `gsis_id` mapping; scheduled team-week and round; source-value reconciliation; and Parquet round trip.

**Diagnostics, not repairs:** Missing `gsis_id`, unusual physical measurements or experience, future-looking `rookie_year`, and incomplete draft-field pairs. Current-season row totals are not hard-coded because 2026 is still in progress.


## 1. Set paths and type contract

The five numeric fields below arrive as whole-valued floats in some partitions. We cast only after proving conversion is exact. Source IDs stay text, including digit-only `gsis_it_id`.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

SEASONS = [2023, 2024, 2025, 2026]
EXPECTED_COLUMNS = 36
ROSTER_KEY = ["gsis_it_id", "season", "week", "team"]
INTEGER_COLUMNS = ["jersey_number", "height", "weight", "rookie_year", "draft_number"]
TEXT_COLUMNS = [
    "team", "position", "depth_chart_position", "status", "full_name",
    "first_name", "last_name", "college", "gsis_id", "espn_id",
    "sportradar_id", "yahoo_id", "rotowire_id", "pff_id", "pfr_id",
    "fantasy_data_id", "sleeper_id", "headshot_url", "ngs_position",
    "game_type", "status_description_abbr", "football_name",
    "esb_id", "gsis_it_id", "smart_id", "draft_club",
]

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "ROADMAP.md").exists():
    assert PROJECT_ROOT != PROJECT_ROOT.parent, "Open this notebook inside the project checkout"
    PROJECT_ROOT = PROJECT_ROOT.parent

BRONZE_DIR = PROJECT_ROOT / "data" / "bronze" / "rosters_weekly"
SCHEDULE_DIR = PROJECT_ROOT / "data" / "silver" / "schedules"
OUTPUT_DIR = PROJECT_ROOT / "data" / "silver" / "rosters_weekly"


## 2. Inspect source schemas and load each season

The schema contract is 36 column names in source order. Source physical numeric types vary by season, so they are normalized below.


In [2]:
source_columns = None
bronze_partitions = {}
source_inventory = []

for season in SEASONS:
    source_path = BRONZE_DIR / f"rosters_weekly_{season}.parquet"
    schedule_path = SCHEDULE_DIR / f"silver_schedules_{season}.parquet"
    assert source_path.exists(), f"Missing Bronze roster input: {source_path}"
    assert schedule_path.exists(), f"Missing schedule validation input: {schedule_path}"

    column_names = pq.read_schema(source_path).names
    assert len(column_names) == EXPECTED_COLUMNS, (season, len(column_names))
    assert set(TEXT_COLUMNS + INTEGER_COLUMNS + ["season", "week", "years_exp", "entry_year", "birth_date"]) == set(column_names)
    if source_columns is None:
        source_columns = column_names
    else:
        assert column_names == source_columns, f"Schema drift in {source_path}"

    bronze = pd.read_parquet(source_path)
    assert bronze.columns.tolist() == source_columns
    assert bronze["season"].eq(season).all(), f"Unexpected season in {source_path}"
    bronze_partitions[season] = bronze
    source_inventory.append({
        "season": season, "bronze_rows": len(bronze),
        "columns": len(bronze.columns), "source": str(source_path.relative_to(PROJECT_ROOT)),
    })

pd.DataFrame(source_inventory)


,season,bronze_rows,columns,source
0,2023,45655,36,data/bronze/rosters_weekly/rosters_weekly_2023...
1,2024,46579,36,data/bronze/rosters_weekly/rosters_weekly_2024...
2,2025,46849,36,data/bronze/rosters_weekly/rosters_weekly_2025...
3,2026,15314,36,data/bronze/rosters_weekly/rosters_weekly_2026...


## 3. Remove exact duplicates and validate the roster key

A source `gsis_it_id` may have a null `gsis_id`. We retain it. A second, different row for the same roster key is a conflict, not a reason to select a preferred row.


In [3]:
deduplicated_partitions = {}
duplicate_counts = {}

for season in SEASONS:
    bronze = bronze_partitions[season]
    exact_duplicates = bronze.duplicated(keep="first")
    duplicate_counts[season] = int(exact_duplicates.sum())
    source = bronze.loc[~exact_duplicates].reset_index(drop=True)
    assert source[ROSTER_KEY].notna().all().all(), season
    assert not source.duplicated(ROSTER_KEY).any(), (
        season, source.loc[source.duplicated(ROSTER_KEY, keep=False), ROSTER_KEY].head(10)
    )
    assert len(source) == len(bronze) - duplicate_counts[season]
    deduplicated_partitions[season] = source

pd.DataFrame([
    {"season": season, "exact_duplicates_removed": duplicate_counts[season],
     "rows_after_deduplication": len(deduplicated_partitions[season])}
    for season in SEASONS
])


,season,exact_duplicates_removed,rows_after_deduplication
0,2023,0,45655
1,2024,0,46579
2,2025,0,46849
3,2026,0,15314


## 4. Normalize source types and reconcile values

A valid conversion can change storage type, not the underlying source number or null pattern. Birth dates already arrive as datetimes and remain unchanged.


In [4]:
silver_partitions = {}

for season in SEASONS:
    source = deduplicated_partitions[season]
    silver = source.copy()

    for column in TEXT_COLUMNS:
        silver[column] = silver[column].astype("string")
    for column in INTEGER_COLUMNS:
        numeric_values = pd.to_numeric(silver[column], errors="raise")
        populated = numeric_values.dropna()
        assert np.isfinite(populated).all(), (season, column, "non-finite")
        assert populated.mod(1).eq(0).all(), (season, column, "fractional")
        assert populated.abs().le(2**53).all(), (season, column, "unsafe float integer")
        silver[column] = numeric_values.astype("Int64")

    assert pd.api.types.is_datetime64_any_dtype(silver["birth_date"])
    assert silver.columns.tolist() == source_columns
    assert len(silver) == len(source)

    unchanged_columns = [
        column for column in source_columns
        if column not in TEXT_COLUMNS + INTEGER_COLUMNS
    ]
    pd.testing.assert_frame_equal(silver[unchanged_columns], source[unchanged_columns])
    for column in TEXT_COLUMNS:
        pd.testing.assert_series_equal(
            silver[column], source[column].astype("string"), check_names=False
        )
    for column in INTEGER_COLUMNS:
        pd.testing.assert_series_equal(
            silver[column].astype("Float64"),
            pd.to_numeric(source[column]).astype("Float64"),
            check_names=False,
        )

    silver_partitions[season] = silver

print("All 36 source columns and values reconciled after type normalization.")


All 36 source columns and values reconciled after type normalization.


## 5. Validate roster context and identity consistency

The source includes inactive, reserve, and practice-squad statuses. We validate scheduled team-week context, but do not infer snaps or game participation from a roster status.


In [5]:
all_rosters = pd.concat(silver_partitions.values(), ignore_index=True)
assert not all_rosters.duplicated(ROSTER_KEY).any()
assert all_rosters["gsis_it_id"].notna().all()
assert all_rosters["season"].isin(SEASONS).all()
assert all_rosters["week"].between(1, 23).all()
assert all_rosters["game_type"].isin(["REG", "WC", "DIV", "CON", "SB"]).all()
assert all_rosters["status"].notna().all()
assert all_rosters["position"].notna().all()

id_mapping_counts = (
    all_rosters.dropna(subset=["gsis_id"])
    .groupby("gsis_it_id")["gsis_id"]
    .nunique()
)
assert id_mapping_counts.le(1).all(), "One gsis_it_id maps to multiple populated gsis_id values"

schedule_checks = []
for season in SEASONS:
    schedules = pd.read_parquet(
        SCHEDULE_DIR / f"silver_schedules_{season}.parquet",
        columns=["game_id", "season", "week", "game_type", "home_team", "away_team"],
    )
    assert schedules["game_id"].notna().all()
    assert not schedules.duplicated("game_id").any()
    home_weeks = schedules[["season", "week", "game_type", "home_team"]].rename(columns={"home_team": "team"})
    away_weeks = schedules[["season", "week", "game_type", "away_team"]].rename(columns={"away_team": "team"})
    scheduled_team_weeks = pd.concat([home_weeks, away_weeks], ignore_index=True)
    assert not scheduled_team_weeks.duplicated(["season", "week", "team"]).any()

    roster_team_weeks = silver_partitions[season][["season", "week", "game_type", "team"]].drop_duplicates()
    checked = roster_team_weeks.merge(
        scheduled_team_weeks, on=["season", "week", "team"],
        how="left", suffixes=("", "_schedule"), validate="one_to_one", indicator=True,
    )
    assert checked["_merge"].eq("both").all(), season
    assert checked["game_type"].eq(checked["game_type_schedule"]).all(), season
    schedule_checks.append({"season": season, "matched_roster_team_weeks": len(checked)})

pd.DataFrame(schedule_checks)


,season,matched_roster_team_weeks
0,2023,570
1,2024,570
2,2025,570
3,2026,186


## 6. Review source exceptions without repairs

These are source-quality diagnostics, not row filters. A null `gsis_id` is retained alongside its populated roster `gsis_it_id`. Zero weight, future-looking rookie year, and mismatched draft fields are not silently changed.


In [6]:
diagnostics = []
for season in SEASONS:
    silver = silver_partitions[season]
    diagnostics.append({
        "season": season,
        "missing_gsis_id": int(silver["gsis_id"].isna().sum()),
        "zero_weight": int(silver["weight"].eq(0).sum()),
        "unusual_weight": int((silver["weight"].lt(120) | silver["weight"].gt(500)).sum()),
        "years_exp_over_30": int(silver["years_exp"].gt(30).sum()),
        "rookie_year_after_season": int(silver["rookie_year"].gt(silver["season"]).sum()),
        "incomplete_draft_pair": int(silver["draft_club"].isna().ne(silver["draft_number"].isna()).sum()),
    })

diagnostic_summary = pd.DataFrame(diagnostics)
diagnostic_summary


,season,missing_gsis_id,zero_weight,unusual_weight,years_exp_over_30,rookie_year_after_season,incomplete_draft_pair
0,2023,5,74,74,0,11,0
1,2024,7,47,48,0,0,0
2,2025,18,0,0,3,0,17
3,2026,7,0,0,0,0,0


## 7. Persist and verify each Silver partition

All partitions have passed in-memory checks before the first write. Reloaded Parquet must reproduce the full source-shaped Silver frame.


In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
validation_rows = []

for season in SEASONS:
    silver = silver_partitions[season]
    output_path = OUTPUT_DIR / f"silver_rosters_weekly_{season}.parquet"
    silver.to_parquet(output_path, index=False)
    reloaded = pd.read_parquet(output_path)
    pd.testing.assert_frame_equal(reloaded, silver)
    assert reloaded.columns.tolist() == source_columns
    assert len(reloaded.columns) == EXPECTED_COLUMNS
    assert not reloaded.duplicated(ROSTER_KEY).any()
    validation_rows.append({
        "season": season,
        "bronze_rows": len(bronze_partitions[season]),
        "exact_duplicates_removed": duplicate_counts[season],
        "silver_rows": len(reloaded),
        "columns": len(reloaded.columns),
        "output": str(output_path.relative_to(PROJECT_ROOT)),
    })

validation_summary = pd.DataFrame(validation_rows)
assert validation_summary["silver_rows"].eq(
    validation_summary["bronze_rows"] - validation_summary["exact_duplicates_removed"]
).all()
validation_summary


,season,bronze_rows,exact_duplicates_removed,silver_rows,columns,output
0,2023,45655,0,45655,36,data/silver/rosters_weekly/silver_rosters_week...
1,2024,46579,0,46579,36,data/silver/rosters_weekly/silver_rosters_week...
2,2025,46849,0,46849,36,data/silver/rosters_weekly/silver_rosters_week...
3,2026,15314,0,15314,36,data/silver/rosters_weekly/silver_rosters_week...
